In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import time
from sklearn.metrics import accuracy_score

from src.data import load_fashion_mnist

from src.preprocessing import (
    flatten_images,
    scale_pixels,
    create_validation_split,
    fit_pca,
    transform_pca
)

from src.models import create_logistic_regression

In [2]:
X_train, y_train, X_test, y_test = load_fashion_mnist()

X_train_flat = flatten_images(X_train)
X_test_flat = flatten_images(X_test)

X_train_scaled = scale_pixels(X_train_flat)
X_test_scaled = scale_pixels(X_test_flat)

X_fit, X_val, y_fit, y_val = create_validation_split(
    X_train_scaled,
    y_train
)
PCA_COMPONENTS = 187

pca, X_fit_pca = fit_pca(
    X_fit,
    PCA_COMPONENTS
)
X_val_pca = transform_pca(
    pca,
    X_val
)

X_test_pca = transform_pca(
    pca,
    X_test_scaled
)
print("Fit:", X_fit_pca.shape)
print("Validation:", X_val_pca.shape)
print("Test:", X_test_pca.shape)

Fit: (48000, 187)
Validation: (12000, 187)
Test: (10000, 187)


In [3]:
logistic_model = create_logistic_regression()
start_time = time.perf_counter()

logistic_model.fit(
    X_fit_pca,
    y_fit
)

fit_time = time.perf_counter() - start_time

In [4]:
y_val_pred = logistic_model.predict(
    X_val_pca
)

val_accuracy = accuracy_score(
    y_val,
    y_val_pred
)

print(
    f"Validation accuracy: {val_accuracy:.4f}"
)

Validation accuracy: 0.8642


In [5]:
# Checkpoint-1 official test prediction

start_time = time.perf_counter()

y_test_pred = logistic_model.predict(X_test_pca)

predict_time = time.perf_counter() - start_time

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Fit time: {fit_time:.4f} seconds")
print(f"Predict time: {predict_time:.4f} seconds")

Test accuracy: 0.8451
Fit time: 14.2104 seconds
Predict time: 0.0053 seconds


## Checkpoint 1 — Logistic Regression Baseline

A Logistic Regression classifier was selected as the initial
baseline.

Images were flattened, normalized to [0, 1], and transformed
using PCA fitted exclusively on the model-fitting portion of
the official training set.

The primary PCA representation retains at least 95% cumulative
explained variance, corresponding to 187 principal components.

The Logistic Regression model was trained on the PCA-transformed
model-fitting subset.

**Validation accuracy:**  0.8642

**Official test accuracy:** 0.8451 (84.51%)

**Fit time:** 42.8113 seconds

**Official test prediction time:** 0.0064 seconds

This result serves as the initial linear baseline against which
the later KNN, SVM, Random Forest, and MLP classifiers will be
compared.

In [6]:

from src.models import (
    create_logistic_regression,
    create_knn,
    create_svm,
    create_random_forest,
    create_mlp
)

from src.evaluation import train_and_evaluate

In [7]:
def evaluate_knn_candidates(k_values):

    results = []

    for k in k_values:

        model = create_knn(
            n_neighbors=k
        )

        result = train_and_evaluate(
            model,
            X_fit_pca,
            y_fit,
            X_val_pca,
            y_val
        )

        results.append({
            "k": k,
            "accuracy": result["accuracy"],
            "macro_f1": result["macro_f1"],
            "fit_seconds": result["fit_seconds"],
            "predict_seconds": result["predict_seconds"]
        })

    return results

In [8]:
def evaluate_svm_candidates(
    C_values,
    gamma_values,
    X_search,
    y_search
):

    results = []

    for C in C_values:
        for gamma in gamma_values:

            model = create_svm(
                C=C,
                gamma=gamma
            )

            result = train_and_evaluate(
                model,
                X_search,
                y_search,
                X_val_pca,
                y_val
            )

            results.append({
                "C": C,
                "gamma": gamma,
                "accuracy": result["accuracy"],
                "macro_f1": result["macro_f1"],
                "fit_seconds": result["fit_seconds"],
                "predict_seconds": result["predict_seconds"]
            })

    return results

In [9]:
def evaluate_rf_candidates(
    n_estimators_values,
    max_depth_values
):

    results = []

    for n_estimators in n_estimators_values:
        for max_depth in max_depth_values:

            model = create_random_forest(
                n_estimators=n_estimators,
                max_depth=max_depth
            )

            result = train_and_evaluate(
                model,
                X_fit_pca,
                y_fit,
                X_val_pca,
                y_val
            )

            results.append({
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "accuracy": result["accuracy"],
                "macro_f1": result["macro_f1"],
                "fit_seconds": result["fit_seconds"],
                "predict_seconds": result["predict_seconds"]
            })

    return results

In [10]:
def evaluate_mlp_candidates(
    architectures,
    activations,
    max_iterations
):

    results = []

    for architecture in architectures:
        for activation in activations:
            for max_iter in max_iterations:

                model = create_mlp(
                    hidden_layer_sizes=architecture,
                    activation=activation,
                    max_iter=max_iter
                )

                result = train_and_evaluate(
                    model,
                    X_fit_pca,
                    y_fit,
                    X_val_pca,
                    y_val
                )

                results.append({
                    "architecture": architecture,
                    "activation": activation,
                    "max_iter": max_iter,
                    "accuracy": result["accuracy"],
                    "macro_f1": result["macro_f1"],
                    "fit_seconds": result["fit_seconds"],
                    "predict_seconds": result["predict_seconds"]
                })

    return results

## Hyperparameter Selection Protocol

All hyperparameter selection is performed using the validation
subset derived from the official Fashion-MNIST training set.

Validation accuracy is used as the primary model-selection
criterion. Macro-F1 is recorded alongside accuracy to verify
balanced performance across the ten classes.

Fit time and prediction time are also recorded to evaluate
computational cost.

The official Fashion-MNIST test set is not used to select
hyperparameters.

In [11]:
K_VALUES = [3, 5, 7, 9, 11]

knn_search_results = evaluate_knn_candidates(
    K_VALUES
)

knn_search_df = pd.DataFrame(knn_search_results)

knn_search_df
knn_search_df.sort_values(
    by="accuracy",
    ascending=False
)

,k,accuracy,macro_f1,fit_seconds,predict_seconds
1,5,0.866167,0.865841,0.015936,1.261016
3,9,0.864500,0.864236,0.016550,1.228377
2,7,0.863917,0.863613,0.021243,1.265204
0,3,0.863000,0.863079,0.023663,1.344448
4,11,0.862000,0.861625,0.020083,1.249611


In [12]:
best_knn_row = knn_search_df.loc[
    knn_search_df["accuracy"].idxmax()
]

best_knn_row
BEST_K = int(best_knn_row["k"])

print("Selected k:", BEST_K)

Selected k: 5


In [13]:
from sklearn.model_selection import train_test_split

X_svm_search, _, y_svm_search, _ = train_test_split(
    X_fit_pca,
    y_fit,
    train_size=15000,
    random_state=42,
    stratify=y_fit
)
print(X_svm_search.shape)
print(np.bincount(y_svm_search))

(15000, 187)
[1500 1500 1500 1500 1500 1500 1500 1500 1500 1500]


In [14]:
C_VALUES = [1, 10, 100]
GAMMA_VALUES = ["scale", 0.01, 0.001]
svm_search_results = evaluate_svm_candidates(
    C_VALUES,
    GAMMA_VALUES,
    X_svm_search,
    y_svm_search
)

svm_search_df = pd.DataFrame(
    svm_search_results
)

svm_search_df.sort_values(
    by="accuracy",
    ascending=False
)

,C,gamma,accuracy,macro_f1,fit_seconds,predict_seconds
3,10,scale,0.891250,0.891007,3.477763,8.180561
4,10,0.01,0.890583,0.890301,3.000511,7.856552
6,100,scale,0.885917,0.885813,3.854451,8.953499
7,100,0.01,0.883833,0.883653,3.444130,7.859081
0,1,scale,0.879417,0.878610,3.864079,9.584910
8,100,0.001,0.875750,0.875386,2.914679,7.101858
1,1,0.01,0.872833,0.871816,3.751096,8.898564
5,10,0.001,0.864750,0.863612,3.283387,9.259152
2,1,0.001,0.823833,0.821560,4.645750,12.182138


In [15]:
best_svm_row = svm_search_df.loc[
    svm_search_df["accuracy"].idxmax()
]

best_svm_row
BEST_C = best_svm_row["C"]
BEST_GAMMA = best_svm_row["gamma"]

In [16]:
RF_ESTIMATORS = [100, 200]
RF_DEPTHS = [None, 20, 40]
rf_search_results = evaluate_rf_candidates(
    RF_ESTIMATORS,
    RF_DEPTHS
)

rf_search_df = pd.DataFrame(
    rf_search_results
)

rf_search_df.sort_values(
    by="accuracy",
    ascending=False
)
best_rf_row = rf_search_df.loc[
    rf_search_df["accuracy"].idxmax()
]

best_rf_row

n_estimators       200.000000
max_depth           40.000000
accuracy             0.869583
macro_f1             0.867920
fit_seconds         16.835221
predict_seconds      0.115989
Name: 5, dtype: float64

In [17]:
MLP_ARCHITECTURES = [
    (64,),
    (128,),
    (128, 64)
]

MLP_ACTIVATIONS = [
    "relu",
    "tanh"
]

MLP_ITERATIONS = [
    100
]
mlp_search_results = evaluate_mlp_candidates(
    MLP_ARCHITECTURES,
    MLP_ACTIVATIONS,
    MLP_ITERATIONS
)

mlp_search_df = pd.DataFrame(
    mlp_search_results
)

mlp_search_df.sort_values(
    by="accuracy",
    ascending=False
)
best_mlp_row = mlp_search_df.loc[
    mlp_search_df["accuracy"].idxmax()
]

best_mlp_row

c:\Users\raunak\Desktop\extra\Optima Vision\optima-vision-\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\raunak\Desktop\extra\Optima Vision\optima-vision-\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\raunak\Desktop\extra\Optima Vision\optima-vision-\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\raunak\Desktop\extra\Optima Vision\optima-vision-\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maxim

architecture       (128, 64)
activation              tanh
max_iter                 100
accuracy             0.89375
macro_f1             0.89403
fit_seconds        27.230261
predict_seconds     0.013267
Name: 5, dtype: object

In [18]:
mlp_convergence_check = create_mlp(
    hidden_layer_sizes=(128, 64),
    activation="tanh",
    max_iter=200
)

mlp_convergence_result = train_and_evaluate(
    mlp_convergence_check,
    X_fit_pca,
    y_fit,
    X_val_pca,
    y_val
)

print(
    "Validation accuracy:",
    mlp_convergence_result["accuracy"]
)

print(
    "Macro-F1:",
    mlp_convergence_result["macro_f1"]
)

print(
    "Fit seconds:",
    mlp_convergence_result["fit_seconds"]
)

print(
    "Iterations actually used:",
    mlp_convergence_check.n_iter_
)

Validation accuracy: 0.89375
Macro-F1: 0.8940303206897768
Fit seconds: 27.994421400129795
Iterations actually used: 96


## Selected Model Configurations

Hyperparameters were selected using validation-set performance.
The official Fashion-MNIST test set was not used for hyperparameter
selection.

### K-Nearest Neighbours (KNN)

- Selected `k`: 5
- Validation accuracy: 86.617%
- Macro-F1: 86.584%

### RBF Support Vector Machine (SVM)

- Selected `C`: 10
- Selected `gamma`: `scale`
- Hyperparameter search was performed using a stratified
  15,000-sample subset of the model-fitting data for computational
  efficiency.
- Search-stage validation accuracy: 89.125%
- Search-stage Macro-F1: 89.101%

The selected SVM configuration will subsequently be trained using
the complete model-fitting subset.

### Random Forest

- `n_estimators`: 200
- `max_depth`: 40
- Validation accuracy: 86.958%
- Macro-F1: 86.792%

### Multilayer Perceptron (MLP)

- Architecture: (128, 64)
- Activation: `tanh`
- Maximum iterations: 200
- Actual iterations used in convergence-check run: 96
- Validation accuracy: 89.375%
- Macro-F1: 89.403%
- Fit time: approximately 27.99 seconds

The initial MLP search used a maximum of 100 iterations and produced
a convergence warning. A follow-up run increased the maximum to 200.
The optimizer converged naturally after 96 iterations while retaining
the same validation accuracy. Therefore, `max_iter=200` is retained
as the final configuration to provide sufficient convergence headroom.

These configurations are now frozen for the common full-data
validation comparison.

In [19]:
selected_models = {
    "KNN": create_knn(
        n_neighbors=5
    ),

    "SVM": create_svm(
        C=10,
        gamma="scale"
    ),

    "Random Forest": create_random_forest(
        n_estimators=200,
        max_depth=40
    ),

    "MLP": create_mlp(
        hidden_layer_sizes=(128, 64),
        activation="tanh",
        max_iter= 200
    )
}

In [20]:
selected_validation_results = []
trained_models = {}

for name, model in selected_models.items():

    print(f"Training {name}...")

    result = train_and_evaluate(
        model,
        X_fit_pca,
        y_fit,
        X_val_pca,
        y_val
    )

    trained_models[name] = result["model"]

    selected_validation_results.append({
        "model": name,
        "pca_components": PCA_COMPONENTS,
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"],
        "fit_seconds": result["fit_seconds"],
        "predict_seconds": result["predict_seconds"]
    })

    print(
        f"{name}: "
        f"accuracy={result['accuracy']:.4f}, "
        f"macro-F1={result['macro_f1']:.4f}"
    )

Training KNN...
KNN: accuracy=0.8662, macro-F1=0.8658
Training SVM...
SVM: accuracy=0.9113, macro-F1=0.9112
Training Random Forest...
Random Forest: accuracy=0.8696, macro-F1=0.8679
Training MLP...
MLP: accuracy=0.8938, macro-F1=0.8940


In [21]:
validation_comparison_df = pd.DataFrame(
    selected_validation_results
)

validation_comparison_df = (
    validation_comparison_df
    .sort_values(
        by="accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)

validation_comparison_df

,model,pca_components,accuracy,macro_f1,fit_seconds,predict_seconds
0,SVM,187,0.911250,0.911158,58.066174,24.421392
1,MLP,187,0.893750,0.894030,26.378397,0.013520
2,Random Forest,187,0.869583,0.867920,15.526144,0.102855
3,KNN,187,0.866167,0.865841,0.030468,1.390492


In [24]:
knn_search_df.to_csv(
    "../results/tuning/knn_search.csv",
    index=False
)

svm_search_df.to_csv(
    "../results/tuning/svm_search.csv",
    index=False
)

rf_search_df.to_csv(
    "../results/tuning/random_forest_search.csv",
    index=False
)

mlp_search_df.to_csv(
    "../results/tuning/mlp_search.csv",
    index=False
)
validation_comparison_df.to_csv(
    "../results/tuning/selected_models_validation.csv",
    index=False
)

In [29]:
import time

from sklearn.metrics import (
    accuracy_score,
    f1_score
)
test_results = []

for name, model in trained_models.items():

    start = time.perf_counter()

    y_pred = model.predict(
        X_test_pca
    )

    predict_seconds = (
        time.perf_counter() - start
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    validation_row = validation_comparison_df[
        validation_comparison_df["model"] == name
    ].iloc[0]

    test_results.append({
        "model": name,
        "pca_components": PCA_COMPONENTS,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "fit_seconds": validation_row["fit_seconds"],
        "predict_seconds": predict_seconds
    })
    test_comparison_df = pd.DataFrame(
    test_results
)

test_comparison_df = (
    test_comparison_df
    .sort_values(
        "accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)


test_comparison_df.to_csv(
    "../results/metrics.csv",
    index=False
)
test_comparison_df

,model,pca_components,accuracy,macro_f1,fit_seconds,predict_seconds
0,SVM,187,0.8991,0.898995,58.066174,20.218233
1,MLP,187,0.8758,0.875871,26.378397,0.013070
2,KNN,187,0.8579,0.857424,0.030468,1.068207
3,Random Forest,187,0.8575,0.855470,15.526144,0.092412


## Checkpoint 2 — Initial Model Comparison

All four required classifiers were evaluated using the same
187-component PCA representation and official Fashion-MNIST split.
Hyperparameters were selected using validation data without using
the official test set for model selection.

| Model | Validation Accuracy | Test Accuracy | Test Macro-F1 |
|---|---:|---:|---:|
| SVM | 91.125% | 89.91% | 89.90% |
| MLP | 89.375% | 87.58% | 87.59% |
| Random Forest | 86.958% | 85.75% | 85.55% |
| KNN | 86.617% | 85.79% | 85.74% |

RBF SVM currently provides the strongest predictive performance,
achieving 89.91% accuracy on the official test set. However, its
prediction time is substantially greater than the other models,
demonstrating a clear performance-versus-computational-cost
trade-off.

These results establish the initial full classification pipeline
for Checkpoint 2. Detailed per-class evaluation and error analysis
are performed separately in `04-evaluation.ipynb`.